# Phase gates and rotations
**Quantum Computing · Week 2 (Gates and the Bloch sphere) · Day 4**

Lesson page: https://sara-aminpour.com/tutorials/qc/w2d4.html

In [ ]:
# On Google Colab, this installs qiskit (about a minute). On your own computer, skip it and
# install the packages once instead: pip install qiskit==2.5.2 scikit-learn torch matplotlib
import sys
if "google.colab" in sys.modules:
    %pip install --quiet qiskit==2.5.2

Every one-qubit gate is a 2×2 matrix that keeps states normalized. On the Bloch sphere,
every such gate is simply a **rotation**.

In [ ]:
import numpy as np
from qiskit import QuantumCircuit
from qiskit.quantum_info import Operator, Statevector, Pauli

def show(matrix):
    """Print a small complex matrix neatly."""
    for row in matrix:
        cells = []
        for z in row:
            z = complex(np.round(z, 3))
            cells.append(f"{z.real:+.3f}" if abs(z.imag) < 1e-9 else f"{z.real:+.3f}{z.imag:+.3f}i")
        print("   [ " + "   ".join(cells) + " ]")

for name in ["x", "z", "h", "s"]:
    qc = QuantumCircuit(1)
    getattr(qc, name)(0)                     # the same as qc.x(0), qc.z(0), ...
    print(f"{name.upper()} =")
    show(Operator(qc).data)

## Follow the arrow
The Bloch vector has coordinates $(\langle X\rangle, \langle Y\rangle, \langle Z\rangle)$, the average
results of three different measurements (more about that in week 4). Let's watch it move.

In [ ]:
def bloch_vector(sv):
    return [round(float(np.real(sv.expectation_value(Pauli(p)))), 3) for p in "XYZ"]

state = Statevector.from_label("0")
print("start      :", bloch_vector(state))
for gate in ["h", "s", "s", "h"]:
    qc = QuantumCircuit(1)
    getattr(qc, gate)(0)
    state = state.evolve(qc)
    print(f"after {gate.upper():<5}:", bloch_vector(state))

Up the north pole → H → the +x side → S → the +y side → S → the −x side → H → the south pole.
So H·S·S·H turned $|0\rangle$ into $|1\rangle$, just like an X gate.

In [ ]:
def same(a, b):
    return Operator(a).equiv(Operator(b))    # equal up to a global phase

hh = QuantumCircuit(1); hh.h(0); hh.h(0)
hzh = QuantumCircuit(1); hzh.h(0); hzh.z(0); hzh.h(0)
x_gate = QuantumCircuit(1); x_gate.x(0)
nothing = QuantumCircuit(1)

print("H then H  == do nothing ?", same(hh, nothing))
print("H, Z, H   == X ?         ", same(hzh, x_gate))

## Rotation gates have a knob
`RY(θ)` tilts the arrow from the north pole towards the south pole by an angle θ.
A Qiskit `Parameter` lets us build the circuit once and fill in θ later.

In [ ]:
import matplotlib.pyplot as plt
from qiskit.circuit import Parameter

theta = Parameter("θ")
qc = QuantumCircuit(1)
qc.ry(theta, 0)

angles = np.linspace(0, 2 * np.pi, 61)
p1 = [Statevector(qc.assign_parameters({theta: a})).probabilities()[1] for a in angles]

plt.plot(angles, p1, "o", ms=4, label="Qiskit")
plt.plot(angles, np.sin(angles / 2) ** 2, label="sin²(θ/2)")
plt.xlabel("θ (radians)")
plt.ylabel("P(1)")
plt.title(r"RY(θ) applied to $|0\rangle$")
plt.legend()
plt.show()

In [ ]:
qc = QuantumCircuit(1)
qc.h(0)
qc.s(0)
qc.ry(0.6, 0)
qc.rz(1.2, 0)
qc.measure_all()
qc.draw("mpl", style="clifford")

## Your turn: same outputs, same gate?
$RY(\pi/2)$ sends $|0\rangle$ to $|+\rangle$, just like H. Does it act like H on $|1\rangle$? On $|+\rangle$?
Fill in the checks. `equiv` compares two states (or two gates) ignoring a global phase.

In [ ]:
ry = QuantumCircuit(1)
ry.ry(np.pi / 2, 0)
had = QuantumCircuit(1)
had.h(0)
s = 1 / np.sqrt(2)
for name, amps in {"|0>": [1, 0], "|1>": [0, 1], "|+>": [s, s]}.items():
    a = Statevector(amps).evolve(ry)
    b = Statevector(amps).evolve(had)
    print(f"on {name}: same result? {None}")       # TODO: a.equiv(b)
print("same gate?", None)                          # TODO: Operator(ry).equiv(Operator(had))

---
## Solution
Try the exercise yourself first: it's fine to get stuck for a minute. Then compare with this one (there are many good answers).

In [ ]:
for name, amps in {"|0>": [1, 0], "|1>": [0, 1], "|+>": [s, s]}.items():
    a = Statevector(amps).evolve(ry)
    b = Statevector(amps).evolve(had)
    print(f"on {name}: same result? {a.equiv(b)}")
print("same gate?", Operator(ry).equiv(Operator(had)))
fix = QuantumCircuit(1)
fix.z(0)
fix.ry(np.pi / 2, 0)
print("Z then RY(pi/2) equals H?", Operator(fix).equiv(Operator(had)))
print("They agree on |0> and on |1> separately, but RY(pi/2)|1> = -|->: the minus sign is a global phase")
print("for that input alone, yet a *relative* phase once |0> and |1> are mixed. Compare gates as matrices.")